In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
from broharness.llms.bedrock import bedrock, UserMessage, AIMessage, SystemMessage
from broharness.flows.skill_call import SkillCall
from broharness.flows.tool_call import ToolCall
from broharness.flows.ask_user_question import AskUserQuestion
from broharness.flows.fail_recovery import FailRecovery
from broharness.flows.answer import Answer
from broflow import BaseTask, TaskRegistry, Flow
from pathlib import Path
import yaml
import sys
import subprocess
from broskill import SkillControl, ToolControl
from functools import partial
from broharness.toolblock import (
    tool_to_yaml, 
    load_skill_tool, 
    load_skill_extension_tool, 
    load_tool_tool, 
    ask_user_question_tool
)
from broharness.codeblock import parse_json_codeblock
from broharness.data_model import Process, State, LLMUse


ROOT = Path.cwd().resolve().parent
SKILL_DIR = ROOT / "skills"

sc = SkillControl(SKILL_DIR)
sc.list_skills()
tc = ToolControl(sc)

In [3]:
skill_call = SkillCall(name='skill-call', llm=bedrock, system_prompt='')
tool_call = ToolCall(name='tool-call', llm=bedrock, system_prompt='')
ask_user_question = AskUserQuestion(name='ask-user-question', llm=bedrock, system_prompt='')
fail_recovery = FailRecovery(name='fail-recovery', llm=bedrock, system_prompt='')
answer = Answer(name='answer', llm=bedrock, system_prompt='')

In [4]:
registry = TaskRegistry()
registry.register(Process.SKILL_CALL, skill_call)
registry.register(Process.TOOL_CALL, tool_call)
registry.register(Process.ASK_USER_QUESTION, ask_user_question)
registry.register(Process.FAIL_RECOVERY, fail_recovery)
registry.register(Process.ANSWER, answer)

flow = Flow(registry)

In [5]:
# this trigger load_skill with skill_name='tell-jokes'
content = "tell me some jokes."
# this trigger ask_user_question
# content = "What's the capital of France?"
# this trigger nothing
# content = "1+1 is?"
messages = [UserMessage(content)]
state = State(
    messages=messages,
    skill_control=sc,
    tool_control=tc,
    debug=messages.copy()
)
# state = skill_call(state)

_ = flow.run(start=Process.SKILL_CALL, end=Process.END, state=state)

D:\study-on-agent\src\broharness\flows\skill_call.py
D:\study-on-agent\src\broharness\flows\tool_call.py
D:\study-on-agent\src\broharness\flows\ask_user_question.py
D:\study-on-agent\src\broharness\flows\tool_call.py


KeyError: <Process.TOOL_USE: 'tool_use'>

In [6]:
print(state.system_prompt)


Skill name: tell-joke
-----
# Tell Joke

## Instructions

- Ask the user which kind of joke they'd like: dad jokes, puns, or a mix of both.
- If they already said which kind in their request, don't ask again -- just tell one.
- If their answer is unclear, ask a short clarifying question directly in your reply. Call `ask_followup_question`.
- Tell exactly one joke at a time, in your own words. Don't paste a reference file's
  contents back verbatim.
- Keep it short, and don't explain the joke afterward -- a joke that needs explaining isn't funny.

## References

- `references/dad-joke.md` -- use when the user wants a dad joke.
- `references/pun-joke.md` -- use when the user wants a pun.
# Tool Call

Pick tools only -- never a skill name. Skill selection already happened in an
earlier `skill-call` step; the caller appends an `## Available Tools` section
below, built fresh each call from the currently-loaded skill's own tools.

## Instructions

- Pick every tool that matches the request 

In [7]:
state.messages

[{'role': 'user', 'content': [{'text': 'tell me some jokes.'}]},
 {'role': 'assistant',
  'content': [{'text': 'What kind of joke would you like to hear?'}]},
 {'role': 'user', 'content': [{'text': 'Dad joke, please.'}]}]

In [8]:
state.debug

[{'role': 'user', 'content': [{'text': 'tell me some jokes.'}]},
 {'role': 'assistant',
  'content': [{'text': '```json\n{\n  "tool_use": [\n    { "name": "load_skill", "input": { "skill_name": "tell-joke" } }\n  ]\n}\n```'}]},
 {'role': 'assistant',
  'content': [{'text': '```json\n{\n  "tool_use": [\n    {\n      "name": "ask_user_question",\n      "input": {\n        "question": "What kind of joke would you like to hear?"\n      }\n    }\n  ]\n}\n```'}]},
 {'role': 'user', 'content': [{'text': 'Dad joke, please.'}]},
 {'role': 'assistant',
  'content': [{'text': '```json\n{\n  "tool_use": [\n    { "name": "load_skill_extension", "input": { "skill_name": "tell-joke", "path": "references/dad-joke.md" } }\n  ]\n}\n```'}]}]

In [6]:
parse_json_codeblock(state.debug[-1]['content'][0]['text'])

{'tool_use': [{'name': 'ask_user_question',
   'input': {'question': 'What kind of jokes would you like: dad jokes, puns, or a mix of both?'}}]}

In [7]:
state.error_message

'no ```json``` codeblock found in response'

In [8]:
flow.trace

[('skill-call', <Process.TOOL_CALL: 'tool_call'>),
 ('tool-call', <Process.ASK_USER_QUESTION: 'ask_user_question'>),
 ('ask-user-question', <Process.TOOL_CALL: 'tool_call'>),
 ('tool-call', <Process.FAIL_RECOVERY: 'fail_recovery'>),
 ('fail-recovery', <Process.TOOL_CALL: 'tool_call'>),
 ('tool-call', <Process.FAIL_RECOVERY: 'fail_recovery'>),
 ('fail-recovery', <Process.TOOL_CALL: 'tool_call'>),
 ('tool-call', <Process.FAIL_RECOVERY: 'fail_recovery'>),
 ('fail-recovery', <Process.TOOL_CALL: 'tool_call'>),
 ('tool-call', <Process.FAIL_RECOVERY: 'fail_recovery'>),
 ('fail-recovery', <Process.ANSWER: 'answer'>),
 ('answer', <Process.END: 'end'>)]

In [10]:
state.messages.pop(-1)

'```json\n{\n  "tool_use": [\n    { "name": "load_skill", "input": { "skill_name": "dad-jokes" } }\n  ]\n}\n```'

In [11]:
state.messages

[{'role': 'user', 'content': [{'text': 'tell me some jokes.'}]},
 {'role': 'assistant',
  'content': [{'text': 'Would you like dad jokes, puns, or a mix of both?'}]},
 {'role': 'user', 'content': [{'text': 'Dad joke please.'}]}]

In [12]:
state.registered_skills

{'load_skill': "# Tell Joke\n\n## Instructions\n\n- Ask the user which kind of joke they'd like: dad jokes, puns, or a mix of both.\n- If they already said which kind in their request, don't ask again -- just tell one.\n- If their answer is unclear, ask a short clarifying question directly in your reply. Call `ask_followup_question`.\n- Tell exactly one joke at a time, in your own words. Don't paste a reference file's\n  contents back verbatim.\n- Keep it short, and don't explain the joke afterward -- a joke that needs explaining isn't funny.\n\n## References\n\n- `references/dad-joke.md` -- use when the user wants a dad joke.\n- `references/pun-joke.md` -- use when the user wants a pun."}

In [15]:
state = tool_call(state)

D:\study-on-agent\src\broharness\flows\tool_call.py


In [16]:
state.messages

[{'role': 'user', 'content': [{'text': 'tell me some jokes.'}]},
 {'role': 'assistant',
  'content': [{'text': 'Would you like dad jokes, puns, or a mix of both?'}]},
 {'role': 'user', 'content': [{'text': 'Dad joke please.'}]}]

In [17]:
state.debug

[{'role': 'user', 'content': [{'text': 'tell me some jokes.'}]},
 {'role': 'assistant',
  'content': [{'text': '```json\n{\n  "tool_use": [\n    { "name": "load_skill", "input": { "skill_name": "tell-joke" } }\n  ]\n}\n```'}]},
 {'role': 'assistant',
  'content': [{'text': '```json\n{\n  "tool_use": [\n    {\n      "name": "ask_user_question",\n      "input": {\n        "question": "Would you like dad jokes, puns, or a mix of both?"\n      }\n    }\n  ]\n}\n```'}]},
 {'role': 'user', 'content': [{'text': 'Dad joke please.'}]},
 {'role': 'assistant',
  'content': [{'text': "Why don't scientists trust atoms? Because they make up everything!"}]},
 {'role': 'assistant',
  'content': [{'text': "Why don't scientists trust atoms? Because they make up everything!"}]},
 {'role': 'assistant',
  'content': [{'text': "Why don't scientists trust atoms? Because they make up everything!"}]},
 {'role': 'assistant',
  'content': [{'text': "Why don't scientists trust atoms? Because they make up everythi

In [19]:
tool_call.next_action

<Process.FAIL_RECOVERY: 'fail_recovery'>

In [18]:
state.error_message

'no ```json``` codeblock found in response'